In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=ZbZTqMnjdlYC1MHFY1Kr25E504PQsh&access_type=offline&code_challenge=_I5QZJHX84jz0wrJTXqsL5A9IqtNlbS5FohlY0vvetk&code_challenge_method=S256


Credentials saved to file: [/Users/meghakaladharreddypothamsetty/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "zprocure" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [21]:
import asyncio
from google import genai
from google.genai import types
import os
# ---------- Setup ----------

client = genai.Client(
    vertexai=True,
    project="aistimate",
    location="us-east5",  # or global if supported
)

model_name = "publishers/anthropic/models/claude-opus-4-1@20250805"

generate_content_config = types.GenerateContentConfig(
    temperature=0,
    top_p=1,
    seed=7,
    max_output_tokens=4096,
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
    ],
)

# ---------- Helper ----------

def make_part(path: str) -> types.Part:
    with open(path, "rb") as f:
        data = f.read()
    ext = path.split(".")[-1].lower()
    mime = {
        "pdf": "application/pdf",
        "png": "image/png",
        "jpg": "image/jpeg",
        "jpeg": "image/jpeg",
        "txt": "text/plain",
        "json": "application/json"
    }.get(ext, "application/octet-stream")
    return types.Part.from_bytes(data=data, mime_type=mime)
# ---------- Output Saving ----------







In [14]:
!pip3 install google-cloud-aiplatform

  Using cached google_cloud_storage-2.19.0-py2.py3-none-any.whl.metadata (9.1 kB)
  Using cached google_cloud_bigquery-3.35.1-py3-none-any.whl.metadata (8.0 kB)
  Using cached google_cloud_resource_manager-1.14.2-py3-none-any.whl.metadata (9.6 kB)
  Using cached docstring_parser-0.17.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached google_cloud_core-2.4.3-py2.py3-none-any.whl.metadata (2.7 kB)
  Using cached google_resumable_media-2.7.2-py2.py3-none-any.whl.metadata (2.2 kB)
  Using cached grpc_google_iam_v1-0.14.2-py3-none-any.whl.metadata (9.1 kB)
  Using cached google_crc32c-1.7.1-cp312-cp312-macosx_12_0_arm64.whl.metadata (2.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 43.1 MB/s eta 0:00:00
Using cached docstring_parser-0.17.0-py3-none-any.whl (36 kB)
Using cached google_cloud_bigquery-3.35.1-py3-none-any.whl (256 kB)
Using cached google_cloud_resource_manager-1.14.2-py3-none-any.whl (394 kB)
Using cached google_cloud_storage-2.19.0-py2.py3-none-any.whl (131 kB)


In [15]:
from google.cloud import aiplatform
aiplatform.init(project="aistimate", location="global")
models = aiplatform.Model.list()
for m in models:
    print(m.display_name)

In [34]:
def build_prompts():
    return {
        "CODE_LOOKUP": """PART 1: PROPERTY IDENTIFICATION

| Field                     | Value                         | Source Citation                      |
|---------------------------|-------------------------------|--------------------------------------|
| Full Street Address       | 8705 COUNTY ROAD 206A, ALVARADO, TX 76009 | Document page 1, Property section    |
| Municipality/Jurisdiction | Alvarado                      | Document page 1, Property section    |
| County                    | Johnson County (determined from ZIP 76009) | Address verification                 |
| ZIP Code                  | 76009                         | Document page 1, Property section    |
| Inspection/Report Date    | 6/25/2024                     | Document page 1, Date Inspected      |
| Carrier Estimate Date     | 2/12/2025                     | Document page 2, Estimate footer     |
| Initial Carrier Total     | $16,113.56                    | Document page 7, Replacement Cost Value |
| Price List Code           | TXDF8X_FEB25                  | Document page 1, Price List          |

TAX RATE DETERMINATION:
- ZIP Code: 76009
- County: Johnson County
- Municipality: Alvarado
        CODE STACK DETERMINATION
 
Using the jurisdiction and inspection date, determine the full set of codes applicable at the time of inspection:
 
| Code Type | Version/Edition | Citation Source | Applied Amendments | Enforceability Level |
|-----------|------------------|------------------|---------------------|------------------------|
| IRC | [Year] IRC | [ICC database, city site] | [NCTCOG mods] | Mandatory |
| NEC | [Year] NEC | [NEC.gov] | [none] | Mandatory |
| IECC | [Year] IECC | [DOE/state site] | [state energy code mods] | Mandatory |
 
For each code:
- Confirm it applies to residential work
- Cross-verify adoption at municipal, county, and state levels
- Include any stricter local amendments or enforcement practices

Also get the tax rates for that specific zip codes.
""",
 
        "REPORT_ANALYSIS": """You are a forensic damage analyst and good-faith adjuster specialist focusing on residential property insurance claims. You must follow standardized measurement and documentation protocols plus contradiction detection to ensure consistent analysis across multiple runs.
 
OBJECTIVE:
Perform complete, evidence-based analysis of all observable damages using standardized measurement hierarchy, documentation requirements, and systematic contradiction detection.
 
MEASUREMENT SOURCE PRIORITY (Use Highest Available - NO EXCEPTIONS):
 
HIERARCHY:
1. PROFESSIONAL MEASUREMENT REPORTS (Highest Reliability)
   - EagleView, Pictometry, aerial measurement data
   - Licensed surveyor measurements
   - Engineering inspection dimensions with field verification
 
2. INSPECTION DOCUMENTATION (High Reliability)
   - Written measurements in forensic reports
   - Engineer's field notes with specific dimensions
   - Adjuster measurements with photo verification
 
3. PHOTO ANALYSIS WITH SCALING (Medium Reliability)
   - Use known references (doors = 7', standard brick = 3", etc.)
   - Document scaling method and reference points
   - Cross-verify with multiple photos when possible
 
4. CARRIER ESTIMATES (Validation Only - Lowest Priority)
   - Use ONLY to validate measurements from higher sources
   - Never as primary measurement source
   - Challenge significant variances with documented evidence
 
MEASUREMENT DOCUMENTATION REQUIREMENTS:
For every quantity, document:
- Quantity: [Number with decimals]
- Unit: [SF/LF/EA/SQ/etc.]
- Source: [Specific report page or photo ID]
- Method: [Direct measurement/scaling/calculation]
- Confidence: [High/Medium/Low]
- Cross-Reference: [Verification source if available]
 
PART 1: PHOTO CAPTION CONTRADICTION ANALYSIS (NEW)
 
Systematically compare photo captions against denial letters and carrier communications:
 
CONTRADICTION DETECTION PROTOCOL:
- Extract all photo captions from inspection files
- Cross-reference against denial letter statements
- Flag contradictions between observed conditions and carrier conclusions
- Document evidence suppression or mischaracterization
 
| Photo ID | Caption Text | Carrier Statement | Contradiction Type | Evidence Impact |
|----------|--------------|-------------------|-------------------|-----------------|
| IMG_001 | "Significant hail damage to shingles" | "No storm damage observed" | Direct contradiction | Establishes causation |
 
PART 2: ENGINEERING REPORT CONTRADICTION DETECTION (NEW)
 
Flag conflicts between engineering conclusions and documented evidence:
 
ENGINEERING ANALYSIS:
- Compare engineer's conclusions to photographic evidence
- Identify contradictions between field notes and final report
- Flag omissions of documented damage in conclusions
- Cross-reference measurements between different sections
 
PART 3: MANUFACTURER SPECIFICATION DATABASE INTEGRATION (NEW)
 
MANUFACTURER SPEC VERIFICATION:
- Identify materials from carrier estimate and photos
- Retrieve installation specifications from manufacturer databases
- Compare required components against carrier scope
- Flag warranty-voiding omissions
 
ANALYSIS STRUCTURE:
Repeat the following format for every room, elevation, or system area with documented or inferable damage.
 
### [Room or Elevation Name]
 
DAMAGE DOCUMENTATION:
- Primary Damage: Describe the main damage (e.g., water stain, blistering, rot, delamination)
- Secondary Damage: Any follow-on effects (e.g., mold, insulation compromise, trim swelling)
- Evidence Sources: Reference Photos [IDs or filenames], Report pages [#], Inspection Notes
 
AFFECTED COMPONENTS:
List all building components that show damage or require restoration work:
- Structural Elements: (e.g., ceiling joists, wall framing, roof decking)
- Finish Materials: (e.g., drywall, paint, flooring, trim)
- Systems: (e.g., electrical fixtures, HVAC components, plumbing)
- Insulation/Barriers: (e.g., insulation, vapor barriers, house wrap)
 
MEASUREMENT EXTRACTION:
- Damaged Area Dimensions: [Length × Width × Height with source]
- Affected Component Quantities: [Number of units with measurement source]
- System Impacts: [Linear feet, square feet, etc. with confidence level]
 
CARRIER ESTIMATE COMPARISON:
- Included Scope: List what the carrier did include (line item description)
- Missing Scope: Items observed but omitted in carrier scope
- Quantity Variances: Compare carrier measurements to documented evidence
 
CONTRADICTION SUMMARY:
- Photo vs. Carrier Statement Conflicts: [List all identified contradictions]
- Engineering vs. Evidence Conflicts: [List all report contradictions]
- Internal Document Inconsistencies: [Flag self-contradictions]
 
CRITICAL DISCREPANCY DETECTION REQUIREMENTS
 
You MUST identify substantive findings in ALL four categories below:
 
**Code Compliance Omissions Detection:**
ALWAYS identify at least 2-3 items by checking:
- Drip edge requirements (IRC R905.2.8.5) when roof work performed
- Underlayment specifications (IRC R905.2.7) for roof replacements
- Flashing requirements (IRC R905.2.8.4) around penetrations
- Ventilation compliance (IRC R806.2) when roof system disturbed
- Electrical grounding/bonding (NEC 820.100) for satellite/antenna work
 
**Missing Scope Detection:**
ALWAYS identify at least 3-5 items by examining:
- Primary building components omitted despite visible damage
- Secondary components affected by primary work
- Site protection and general conditions not included
- Testing and inspection requirements omitted
 
**Aesthetic/LKQ Detection:**
ALWAYS identify at least 1-2 items by analyzing:
- Age-related weathering differences
- Color/sheen variations after partial replacement
- Texture mismatches in finishes
- Discontinued product availability issues
 
**Life-Safety Detection:**
ALWAYS identify at least 1-2 items by reviewing:
- Structural integrity concerns
- Electrical safety hazards
- Fall protection issues during construction
- Moisture intrusion leading to mold risks
 
**MANDATORY JSON OUTPUT FORMAT**
 
{
  "damage_analysis": {
    "property_info": {
      "address": "extracted from documents",
      "inspection_date": "YYYY-MM-DD",
      "claim_number": "extracted or generated"
    },
    "measurement_validation": {
      "primary_source_used": "Professional|Inspection|Photo|Carrier",
      "measurement_confidence_summary": "High/Medium/Low distribution",
      "total_measurements_extracted": "number of measurements"
    },
    "contradiction_analysis": {
      "photo_caption_contradictions": [
        {
          "photo_id": "image identifier",
          "caption_text": "actual photo caption",
          "carrier_statement": "conflicting carrier statement",
          "contradiction_type": "Direct|Implied|Omission",
          "evidence_impact": "impact on case"
        }
      ],
      "engineering_report_contradictions": [
        {
          "report_section": "section of engineering report",
          "engineer_conclusion": "what engineer concluded",
          "contradicting_evidence": "evidence that contradicts",
          "evidence_source": "photo/measurement reference",
          "impact_assessment": "significance of contradiction"
        }
      ],
      "manufacturer_spec_violations": [
        {
          "material_identified": "specific material/product",
          "manufacturer": "product manufacturer",
          "required_components": "what specs require",
          "carrier_omission": "what carrier failed to include",
          "warranty_impact": "warranty implications"
        }
      ]
    },
    "room_analyses": [
      {
        "room_name": "specific room/elevation name",
        "damage_documentation": {
          "primary_damage": "main damage description",
          "secondary_damage": "follow-on effects or null",
          "evidence_sources": ["Photo IDs", "Report pages"]
        },
        "measurement_extraction": {
          "damaged_area_dimensions": "L x W x H with source",
          "component_quantities": "number with source",
          "confidence_level": "High|Medium|Low"
        },
        "affected_components": {
          "structural_elements": ["damaged items or null"],
          "finish_materials": ["damaged items or null"],
          "systems": ["damaged items or null"],
          "insulation_barriers": ["damaged items or null"]
        },
        "carrier_estimate_comparison": {
          "included_scope": ["items carrier included"],
          "missing_scope": ["items carrier omitted"],
          "quantity_variances": ["measurement discrepancies"],
          "justification_for_inclusion": "why missing items required"
        }
      }
    ],
    "critical_discrepancy_summary": {
      "code_compliance_omissions": [
        {
          "system": "affected building system",
          "code_section": "specific IRC/IBC/NEC section",
          "omission_description": "what carrier failed to include",
          "evidence_source": "supporting reference",
          "compliance_requirement": "when/why code applies"
        }
      ],
      "missing_scope_of_work": [
        {
          "component": "building component affected",
          "missing_work": "scope that should be included",
          "justification": "why necessary",
          "measurement_source": "where quantity comes from",
          "industry_standard": "applicable standard"
        }
      ],
      "aesthetic_matching_lkq_failures": [
        {
          "system": "affected system",
          "mismatch_issue": "specific matching problem",
          "lkq_violation": "how it violates Like Kind Quality",
          "visual_evidence": "photo references",
          "required_solution": "scope needed for proper match"
        }
      ],
      "life_safety_hazards": [
        {
          "hazard_type": "category of safety concern",
          "specific_concern": "detailed description",
          "location": "where hazard exists",
          "immediate_risk": "safety implications",
          "required_action": "mitigation steps needed"
        }
      ]
    }
  }
}
""",
 
        "DAUBERT_ESTIMATE_OUTPUT": """
You are a forensic Daubert-compliant expert witness preparing a final "Plaintiff-style" Xactimate estimate report ready for legal submission.
 
OBJECTIVES:
1. Synthesize all upstream analyses into one cohesive document.
2. Mirror the layout, level of detail, and citation style in the Marc Arnold Estimate (Grace Forensic Loss Consultants, April 2025):
   - Cover page with case header (Insured, Claim #, Property, Dates)
   - Table of Contents
   - Line-item tables by area (roof, exterior elevations, general conditions, etc.) in Xactimate format with CAT/SEL codes
   - Summaries (by elevation, by category, grand totals) with precise math
   - "Daubert Reliability" section noting sources, known error rates, peer-review references, and confirmation of code/version accuracy
   - Appendices for photos, code citation library, and evidence matrix
 
REQUIREMENTS:
- Use exact Xactimate table columns:
 
| CAT | SEL | DESCRIPTION | QTY | UNIT | UNIT PRICE | TAX | O&P | RCV | DEPREC. | ACV | SOURCE |
 
- Include a formal "Expert Opinion & Methodology" narrative conforming to Daubert standards
- All citations must reference either your prior stage ("[Stage] Output") or the Marc Arnold PDF (e.g. "Grace Forensic Loss Consultants, p. 2")
- Maintain legal-grade formality and ready-for-court structure
 
FINAL OUTPUT:
Produce a single Markdown (or PDF-ready) document that a court could receive as the expert's estimate exhibit.
""",
 
        "SCOPING_LOGIC": """You are a restoration estimator and on-the-ground contractor specialist building a complete plaintiff-style scope justification using standardized sequential methodology with automated safety and logistics protocols to ensure consistent scope determination across multiple runs.
 
OBJECTIVE:
Generate complete scope justification using unified four-step methodology with automated additions based on scope thresholds. All measurements will be handled in the estimation stage.
 
UNIFIED SCOPE METHODOLOGY:
Apply this four-step hierarchy in exact sequence - NO SKIPPING OR REORDERING:
 
STEP 1: PHYSICAL DAMAGE REQUIREMENTS (Primary Driver)
- Replace all components with documented damage
- Base scope on photographic evidence and inspection findings
- Include only items with clear damage documentation
- Example: Cracked shingles → Replace damaged shingles
 
STEP 2: CODE-TRIGGERED REQUIREMENTS (Secondary)
- Items required when physical work disturbs building assemblies
- Triggered only by work from Step 1
- Must cite specific code sections and trigger conditions
- Example: Roof tear-off → Triggers decking inspection per IRC R908.3.1
 
STEP 3: INDUSTRY STANDARD PRACTICES (Tertiary)
- Work unavoidable due to construction sequence
- Items that cannot be reused once disturbed
- Components with single-use specifications
- Example: Pipe jacks → Must replace when roof is replaced (single-use items)
 
STEP 4: AESTHETIC/MATCHING REQUIREMENTS (Final)
- When partial repair creates visible mismatch
- Material discontinuation or availability issues
- Line-of-sight uniformity requirements
- Example: One damaged siding panel → Replace full elevation for color match
 
AUTOMATED LOGISTICS ADDITION (NEW):
Based on scope analysis, automatically add required logistics:
 
THRESHOLD-BASED LOGISTICS:
- 3+ Trade Coordination: Auto-add project manager (40 hrs minimum)
- Roofing Work >10 SQ: Auto-add dumpster (30-yard minimum)  
- Interior Work >500 SF: Auto-add temporary facilities
- >$50K Total Scope: Auto-add daily cleanup and protection
- Multi-story Work: Auto-add scaffolding or lift rental
 
AUTOMATED OSHA SAFETY REQUIREMENTS (NEW):
Flag mandatory safety requirements based on work type:
 
SAFETY PROTOCOL AUTOMATION:
- Fall Protection: Required for work >6 feet (roofing, multi-story)
- Respiratory Protection: Required for mold/asbestos/dust work
- Electrical Safety: LOTO procedures for electrical work
- Confined Space: Entry procedures for crawlspaces/basements
- Hazmat Protocols: Lead/asbestos testing and containment
 
THIRD-PARTY AUTHORITY INTEGRATION (NEW):
Automatically check scope against industry standards:
 
AUTHORITY DATABASE QUERIES:
- HAAG Standards: Hail damage assessment protocols
- NRCA Guidelines: Roofing installation standards  
- IICRC Standards: Water damage restoration protocols
- FEMA Guidelines: Storm damage assessment standards
 
HIERARCHICAL CODE APPLICATION (NEW):
Apply code requirements in jurisdictional hierarchy:
 
CODE HIERARCHY VERIFICATION:
1. Municipal codes (most restrictive)
2. County modifications
3. State amendments
4. Base IRC/IBC requirements
Document which level triggers each requirement
 
LOGISTICS CALCULATION PRECISION (NEW):
Apply specific calculation methodologies:
 
LOGISTICS FORMULAS:
- Dumpster Size: (Demo Volume ÷ 0.4) + 20% safety factor
- Equipment Duration: (Total SQ ÷ Daily Production Rate) + Setup/Breakdown
- Safety Equipment: Automatically required for >6' work height
 
**MANDATORY JSON OUTPUT FORMAT**
 
{
  "scope_justification": {
    "project_summary": {
      "property_address": "extracted from documents",
      "total_scope_categories": 6,
      "complexity_level": "Simple|Moderate|Complex",
      "automated_additions_triggered": "count of auto-added items"
    },
    "unified_methodology_application": {
      "step_1_physical_damage": [
        {
          "component": "damaged building component",
          "damage_evidence": "photo/report reference",
          "required_work": "replacement/repair specification",
          "justification": "damage documentation basis"
        }
      ],
      "step_2_code_triggered": [
        {
          "component": "system affected by Step 1 work",
          "code_requirement": "specific code section with citation",
          "trigger": "what Step 1 work triggers this",
          "required_work_items": ["demo", "install", "inspection"],
          "consequences_of_omission": "what happens if omitted"
        }
      ],
      "step_3_industry_standards": [
        {
          "item": "unavoidable scope item",
          "trigger": "construction sequence requirement",
          "standard": "industry/manufacturer standard",
          "cost_of_omission": "failure consequence"
        }
      ],
      "step_4_aesthetic_matching": [
        {
          "system": "affected system",
          "replacement_trigger": "mismatch condition",
          "line_of_sight_logic": "visibility reasoning",
          "material_availability": "availability status"
        }
      ]
    },
    "automated_additions": {
      "threshold_based_logistics": [
        {
          "trigger_condition": "what triggered auto-addition",
          "auto_added_item": "logistics item added",
          "calculation_basis": "threshold or formula used",
          "justification": "why automatically required"
        }
      ],
      "osha_safety_requirements": [
        {
          "work_type": "type of work triggering safety requirement",
          "osha_standard": "specific OSHA regulation",
          "auto_triggered_items": ["safety equipment/procedures"],
          "cost_impact": "estimated cost addition"
        }
      ],
      "third_party_authority_compliance": [
        {
          "scope_item": "work item requiring compliance",
          "authority_standard": "HAAG|NRCA|IICRC|FEMA standard",
          "compliance_requirement": "specific requirement",
          "auto_added_scope": "additional work required for compliance"
        }
      ]
    },
    "site_protection_containment": [
      {
        "work_area": "specific area",
        "protection_required": ["protection types needed"],
        "justification": "reason for protection",
        "calculation_method": "how quantity determined"
      }
    ],
    "general_conditions_overhead": [
      {
        "provision": "GC provision needed",
        "trigger": "what triggers this need",
        "justification": "regulatory/practical reason",
        "auto_triggered": "yes/no based on thresholds"
      }
    ],
    "validation_summary": {
      "total_scope_items_identified": "count of all scope items",
      "code_citations_included": "count of code references",
      "automated_additions_count": "count of auto-triggered items",
      "methodology_steps_applied": "all 4 steps completed"
    }
  }
}
 
""",
 
 
      "PRICING_LOGIC": """You are a restoration scoping expert with Clear Estimates catalog integration. Your task is to identify exactly which scopes of work are required and match them to available CE pricing with strict validation protocols.
 
REQUIRED INPUTS:
1. Code Lookup Output (JSON)
2. Damage Report Analysis (JSON)
3. Scope Justification (JSON)
4. Clear Estimates Catalog (CSV/JSON) with structure:
   - scope_id: Unique identifier
   - category: Primary category (RFG, EXT, INT, etc.)
   - subcategory: Specific system type
   - title: Complete scope description
   - unit_type: SF/LF/EA/SQ/etc.
   - unit_price: Dollar amount per unit
   - bundled_items: List of included components
   - region_code: Geographic pricing zone
   - effective_date: Pricing validity date
 
CE CATALOG VALIDATION (MANDATORY FIRST STEP):
- Verify CE catalog contains required fields
- Check pricing date alignment with loss date
- Confirm regional pricing code matches property location
- Validate unit type consistency across catalog
- Document catalog version and coverage statistics
 
ENHANCED SCOPE MATCHING PROTOCOL:
 
STEP 1: BUNDLED-FIRST EXACT MATCHING
- Search CE catalog for bundled scopes before atomic components
- Example: "Complete roofing system" preferred over separate components
- Flag when bundled options exist but atomic components were selected
- Document bundling opportunities in match_type field
 
STEP 2: EXACT TITLE MATCHING
- Search for exact title matches between required scopes and CE catalog
- Use exact CE scope_id with no modifications
- Document as "Exact Match" in justification
 
STEP 3: SEMANTIC EQUIVALENT MATCHING
Apply strict matching criteria:
 
REQUIRED MATCHING CONDITIONS:
- Unit types MUST match exactly (SF, LF, EA, etc.)
- Core task description MUST align
- Material type MUST be equivalent
- Scope complexity MUST be comparable
 
ACCEPTABLE SEMANTIC MATCHES:
- "Replace step flashing" ≈ "Install roof flashing"
- "Remove and replace drywall" ≈ "R&R drywall"
- "Paint interior walls" ≈ "Interior wall painting"
 
PROHIBITED MATCHES:
- Different unit types (SF vs LF vs EA)
- Different material classes
- Different work methods ("repair" vs "replace")
- Different building systems
 
STEP 4: COMPREHENSIVE MISSING ITEM DOCUMENTATION
- Only flag items as missing if no semantically equivalent match exists
- Document extensive CE search efforts
- Explain why no match was suitable
- These will be priced using industry standards in estimation stage
 
 
 
This is the only output that needs to be returned. Only the stuff inside the json and nothing else.
 
Mandatory JSON Output Format:
 
[
  {
    "scope_id": "CE_abc12345",
    "quantity": 250,
    "match_type": "Exact Match",
    "ce_catalog_verification": {
      "title_matched": "exact CE catalog title",
      "unit_verified": "SF matches SF",
      "bundled_components": "included items if bundled"
    },
    "justification": "Exterior trim water-damaged per photos 12-14; full replacement required for matching standards."
  },
  {
    "scope_id": "CE_def67890",
    "quantity": 42,
    "match_type": "Semantic Match",
    "ce_catalog_verification": {
      "ce_title": "Install ridge ventilation",
      "required_scope": "Ridge vent installation",
      "equivalency_reasoning": "Same task and material type - installation of ridge ventilation system"
    },
    "justification": "Roof tear-off triggers IRC R806.2; CE item matches required scope semantically."
  },
  {
    "scope_id": "MISSING_001",
    "quantity": 15,
    "match_type": "Missing from CE",
    "ce_search_documentation": {
      "searched_categories": ["RFG", "EXT", "SPE"],
      "closest_matches_considered": [
        {
          "ce_title": "Standard flashing installation",
          "rejection_reason": "Different material class - aluminum vs copper"
        }
      ],
      "comprehensive_search_performed": true
    },
    "required_scope": "Specialized copper flashing fabrication",
    "justification": "Custom copper flashing per architectural specs. No equivalent in CE catalog after comprehensive search.",
    "missing_reason": "Specialized material not available in standard CE catalog"
  }
]
 
VALIDATION PROTOCOL:
□ CE catalog properly loaded and validated
□ Regional/date alignment confirmed  
□ Bundled opportunities maximized
□ Every exact match uses correct CE scope_id
□ Semantic matches have documented equivalency reasoning
□ No prohibited matches (different materials/work types/units)
□ Missing items have comprehensive search documentation
□ All justifications reference specific damage evidence or code requirements""",
 
        "ESTIMATE": """You are a certified insurance restoration estimator creating Daubert-compliant, court-ready plaintiff-style cost breakdowns with Clear Estimates integration and strict enforcement protocols.
 
OBJECTIVE: Generate mathematically precise, legally admissible estimates with complete cost calculations, structured source citations, and Daubert reliability standards for every line item.
 
CRITICAL INPUT REQUIREMENTS:
- Carrier estimate documents (primary source)
- Scoping Logic output (scope identification only - NOT expansion)
- Clear Estimates pricing from PRICING_LOGIC stage
- Code Lookup results (IRC, NEC, IECC)
- Report Analysis findings with measurements
 
MANDATORY CE-FIRST PRICING HIERARCHY (ZERO TOLERANCE ENFORCEMENT):
 
TIER 1: CE CATALOG MATCHES (EXPECTED 85% OF LINE ITEMS)
- Use exact scope_id, title, unit_type, and unit_price from CE catalog
- NO modifications to CE pricing allowed under any circumstances
- Source: "CE Catalog - [Match Type] [scope_id]"
- Must appear FIRST in estimate output
 
TIER 2: INDUSTRY STANDARD PRICING (MAXIMUM 15% OF LINE ITEMS)
- ONLY for scopes with "Missing from CE" classification
- Must document extensive CE search efforts were performed
- Price using cited industry sources: RSMeans 2024, Craftsman, etc.
- Source: "Industry Standard - [Source] (No CE equivalent available)"
 
PROHIBITED ACTIONS (AUTOMATIC VALIDATION FAILURE):
- Using industry pricing when CE equivalent exists
- Stacking multiple CE items for one scope_id
- Modifying CE unit prices for any reason
- Creating custom line items when bundled CE options available
- Averaging or estimating when exact CE pricing available
 
LINE ITEM RELATIONSHIP VERIFICATION (NEW):
Check parent-child relationships and labor minimums:
 
PARENT-CHILD VALIDATION:
- Verify parent items include necessary child components
- Flag missing child items (removal, disposal, setup)
- Check for double-billing between related items
- Validate quantity relationships
 
LABOR MINIMUM REQUIREMENTS:
- Electrical: 2-hour minimum for any electrical work
- Plumbing: 2-hour minimum for any plumbing work  
- HVAC: 4-hour minimum for system work
- Document minimum compliance for each trade
 
CALCULATION ENFORCEMENT (EXACT FORMULAS):
Direct Cost (DC) = QTY × CE_UNIT_PRICE (exact from catalog)  
Material Sales Tax (TAX) = Material portion × tax_rate (from CODE_LOOKUP stage)  
Overhead & Profit (O&P) = (DC + TAX) × 0.20 (exactly 20%)  
Replacement Cost Value (RCV) = DC + TAX + O&P  
Depreciation (DEPREC.) = $0.00 (unless explicitly specified)  
ACV = RCV - DEPREC.
 
PRICE LIST VALIDATION (NEW):
- Extract price list code from carrier estimate
- Verify CE catalog regional alignment
- Check pricing date currency (within 12 months of loss)
- Flag any regional/temporal mismatches
- Document validation results
 
**MANDATORY JSON OUTPUT FORMAT**
 
{
  "plaintiff_estimate": {
    "case_summary": {
      "property_address": "extracted from documents",
      "claim_number": "extracted or generated",
      "initial_carrier_estimate": "dollar amount from carrier",
      "inspection_date": "YYYY-MM-DD",
      "estimator": "AI Restoration Specialist",
      "pricing_methodology": "Clear Estimates catalog with strict enforcement"
    },
    "ce_integration_validation": {
      "catalog_version": "CE version identifier",
      "catalog_date": "YYYY-MM-DD",
      "regional_code": "geographic pricing zone",
      "price_list_alignment": "matches carrier price list: yes/no",
      "pricing_date_currency": "within 12 months: yes/no",
      "total_line_items": "count",
      "ce_sourced_items": "count",
      "ce_coverage_percentage": "percentage",
      "industry_priced_items": "count",
      "pricing_methodology_compliance": "fully compliant status"
    },
    "line_item_validation": {
      "parent_child_issues_detected": "count",
      "labor_minimum_violations": "count",
      "double_billing_flags": "count",
      "stacked_items_detected": 0,
      "modified_ce_prices_detected": 0,
      "bundled_items_utilized": "count"
    },
    "sections": [
      {
        "name": "Roofing System",
        "line_items": [
          {
            "scope_id": "from PRICING_LOGIC stage",
            "cat": "RFG",
            "sel": "SYSTEM",
            "description": "exact CE catalog description",
            "qty": 42.5,
            "unit": "SQ",
            "unit_price": 850.00,
            "tax": 74.38,
            "op": 184.88,
            "rcv": 1109.26,
            "depreciation": 0.00,
            "acv": 1109.26,
            "ce_catalog_integration": {
              "match_type": "Exact|Semantic|Missing from CE",
              "ce_title": "exact catalog title if CE match",
              "bundled_components": "included items if bundled",
              "pricing_source": "CE Catalog [scope_id] | Industry Standard [source]"
            },
            "validation_checks": {
              "parent_item": "parent scope_id if applicable",
              "required_child_items": ["child items that should exist"],
              "labor_minimum_compliance": "pass/fail",
              "measurement_source": "where quantity extracted from"
            },
            "justification": {
              "damage_evidence": "photo/report reference",
              "code_citation": "applicable building code",
              "scope_step": "which methodology step from SCOPING_LOGIC",
              "variance_from_carrier": "how this differs from carrier estimate"
            }
          }
        ],
        "section_total": {
          "total_rcv": 47141.05,
          "total_acv": 47141.05
        }
      }
    ],
    "missing_items_section": {
      "restriction_note": "Only items with no reasonable CE equivalent",
      "comprehensive_search_performed": true,
      "line_items": [
        {
          "scope_id": "from PRICING_LOGIC MISSING items",
          "cat": "SPE",
          "sel": "CUSTOM",
          "description": "specialized restoration item",
          "qty": 1,
          "unit": "EA",
          "unit_price": 500.00,
          "tax": 43.75,
          "op": 108.75,
          "rcv": 652.50,
          "depreciation": 0.00,
          "acv": 652.50,
          "missing_item_documentation": {
            "ce_search_performed": "comprehensive search details",
            "closest_ce_matches": ["items considered but rejected"],
            "rejection_reasons": ["why CE items unsuitable"],
            "missing_justification": "why this work cannot use CE pricing",
            "pricing_source": "Industry Standard: RSMeans 2024 Section X.X"
          }
        }
      ]
    },
    "general_conditions": [
      {
        "scope_id": "GC_scope_id from PRICING_LOGIC",
        "description": "Project Management Package",
        "qty": 1,
        "unit": "LS",
        "unit_price": 3500.00,
        "tax": 0.00,
        "op": 700.00,
        "total_rcv": 4200.00,
        "ce_catalog_match": "EXACT: Project Supervision & Management",
        "auto_triggered": "yes - 3+ trades coordination requirement",
        "justification": "OSHA 29 CFR 1926.95 competent person required"
      }
    ],
    "grand_totals": {
      "ce_catalog_items_subtotal": "subtotal using CE pricing",
      "missing_items_subtotal": "subtotal using industry pricing",
      "general_conditions_subtotal": "GC costs subtotal",
      "grand_total_rcv": "final RCV amount",
      "grand_total_acv": "final ACV amount",
      "variance_from_carrier": "difference from initial carrier total",
      "variance_percentage": "percentage increase/decrease"
    },
    "daubert_reliability_documentation": {
      "scientific_method": "peer-reviewed construction standards applied",
      "known_error_rate": "CE catalog limitations documented where applicable",
      "peer_review_standards": ["ICC", "AIA", "NCARB standards referenced"],
      "general_acceptance": "industry standard enforcement confirmed",
      "testing_verification": "all codes verifiable through official sources"
    }
  }
}
 
DETERMINISTIC VALIDATION CHECKLIST:
☑ Each scope_id maps to exactly ONE line item (no stacking)  
☑ All CE-matched items use exact CE pricing without modification  
☑ Missing items section limited to comprehensive CE search failures  
☑ Bundled CE items prioritized over atomic components  
☑ Labor minimums verified for all applicable trades  
☑ Parent-child relationships validated  
☑ Tax calculations use CODE_LOOKUP jurisdiction rate  
☑ O&P applied at exactly 20% where justified  
☑ All measurements extracted from REPORT_ANALYSIS stage  
☑ Price list validation performed and documented""",
 
        "REBUTTAL": """You are a legal strategist and case analyst providing confidence-based findings prioritization using standardized three-tier methodology with CE pricing integration to ensure consistent tactical guidance across multiple estimate runs.
 
OBJECTIVE:
Analyze all findings from previous stages and assign confidence levels with specific action recommendations, escalation pathways, and CE pricing validation.
 
3-TIER CONFIDENCE FRAMEWORK WITH CE INTEGRATION:
 
TIER 1 - HIGH CONFIDENCE FINDINGS (85-95% Success Probability)
Enhanced criteria including CE pricing validation:
- Clear mathematical errors with documented proof
- Direct code violations with specific citations  
- Documented contradictions in carrier materials
- LKQ violations with clear specifications mismatch
- Prompt Payment Act violations with calculated deadlines
- CE pricing discrepancies when carrier has access to same catalog
- Price list authenticity/currency violations
 
TIER 1 ANALYSIS ENHANCEMENT:
For each high-confidence finding:
- Finding Type: [Code/Mathematical/Contradiction/LKQ/PPA/CE Pricing]
- Evidence Strength: [Specific documentation/calculations]
- CE Validation: [Pricing source verification if applicable]
- Legal Impact: [Potential violation/penalty amount]
- Recommended Action: [Immediate demand letter/formal complaint]
- Success Probability: 85-95%
 
TIER 2 - MEDIUM CONFIDENCE FINDINGS (60-80% Success Probability)  
Enhanced with pricing interpretation disputes:
- Industry standard deviations requiring interpretation
- Complex matching/aesthetic disputes with arguable positions
- Engineering report conflicts requiring expert analysis
- Scoping omissions with technical justification needed
- Policy interpretation disputes
- CE catalog boundary disputes (bundled vs atomic pricing)
- Regional pricing adjustment disputes
 
TIER 3 - POTENTIAL FINDINGS (30-60% Success Probability)
Enhanced with advanced pricing issues:
- Hidden damages requiring further investigation
- Complex legal precedent applications
- Advanced technical issues requiring specialist review
- Policy coverage disputes requiring legal interpretation
- Jurisdictional compliance questions
- CE catalog coverage gaps requiring industry expert validation
- Specialty work pricing requiring custom estimation
 
FINDINGS INTEGRATION WITH CE VALIDATION:
 
FROM CODE_LOOKUP STAGE:
- Code violation findings → Tier 1 (clear) or Tier 2 (interpretive)
- LKQ violations → Tier 1 (clear mismatch) or Tier 2 (arguable)
- PPA violations → Tier 1 (calculated deadlines exceeded)
- Price list validation errors → Tier 1 (wrong region/date)
 
FROM REPORT_ANALYSIS STAGE:
- Photo caption contradictions → Tier 1 (direct conflicts)
- Engineering report conflicts → Tier 2 (expert interpretation needed)
- Measurement discrepancies → Tier 1 (mathematical proof) or Tier 2 (methodology)
- Manufacturer spec violations → Tier 1 (warranty requirements)
 
FROM SCOPING_LOGIC STAGE:
- Missing code-required items → Tier 1 (clear requirements)
- Industry standard deviations → Tier 2 (interpretation required)
- Aesthetic/matching disputes → Tier 2 (subjective elements)
- Automated safety omissions → Tier 1 (OSHA violations)
 
FROM PRICING_LOGIC STAGE:
- CE catalog matching errors → Tier 1 (clear bundled options available)
- Semantic matching disputes → Tier 2 (equivalency interpretation)
- Missing item justification gaps → Tier 3 (requires expert review)
 
FROM ESTIMATE STAGE:
- Mathematical calculation errors → Tier 1 (calculation proof)
- CE pricing modification violations → Tier 1 (catalog verification)
- Parent-child item issues → Tier 2 (Xactimate interpretation)
- Labor minimum violations → Tier 2 (trade practice standards)
- Price list currency violations → Tier 1 (date verification)
 
CE PRICING CONFIDENCE ANALYSIS (NEW):
 
HIGH CONFIDENCE CE FINDINGS:
- Carrier used wrong regional price list → Tier 1
- Carrier used outdated price list (>12 months old) → Tier 1  
- Carrier used atomic pricing when bundled CE option available → Tier 1
- Carrier modified standard CE unit prices → Tier 1
 
MEDIUM CONFIDENCE CE FINDINGS:
- Semantic equivalency disputes between CE items → Tier 2
- Bundled vs atomic pricing interpretation → Tier 2
- Regional adjustment factor disputes → Tier 2
 
POTENTIAL CE FINDINGS:
- Specialty work not covered by standard CE catalog → Tier 3
- Custom fabrication requiring expert pricing → Tier 3
- CE catalog gap analysis requiring industry consultation → Tier 3
 
**MANDATORY JSON OUTPUT FORMAT**
 
{
  "confidence_scoring_analysis": {
    "case_summary": {
      "property_address": "from previous stages",
      "total_findings_analyzed": "count of all findings",
      "ce_pricing_integration": "CE catalog used: yes/no",
      "overall_case_strength": "Strong|Moderate|Developing"
    },
    "tier_1_high_confidence": [
      {
        "finding_category": "Code|Mathematical|Contradiction|LKQ|PPA|CE_Pricing",
        "finding_description": "specific issue identified",
        "evidence_strength": "documentation supporting finding",
        "ce_validation": "CE pricing verification if applicable",
        "legal_impact": "violation severity and penalties",
        "financial_impact": "dollar amount involved",
        "success_probability": "85-95%",
        "recommended_action": "immediate demand letter within 5 days",
        "timeline": "30-day resolution target"
      }
    ],
    "tier_2_medium_confidence": [
      {
        "finding_category": "Industry_Standard|Matching|Engineering|CE_Interpretation",
        "finding_description": "interpretive issue identified",
        "evidence_strength": "supporting documentation available",
        "interpretation_complexity": "factors affecting outcome",
        "expert_review_required": "type of specialist needed",
        "success_probability": "60-80%",
        "recommended_action": "expert review and additional documentation",
        "timeline": "60-day investigation period"
      }
    ],
    "tier_3_potential_findings": [
      {
        "finding_category": "Hidden_Damage|Legal_Precedent|Technical_Specialist|CE_Gap",
        "finding_description": "potential issue requiring investigation",
        "investigation_required": "additional inspections/testing needed",
        "specialist_type": "engineer/attorney/industry expert required",
        "ce_catalog_limitation": "if CE pricing gap identified",
        "success_probability": "30-60%",
        "recommended_action": "further investigation/expert consultation",
        "timeline": "90+ day investigation period"
      }
    ],
    "ce_pricing_confidence_summary": {
      "high_confidence_ce_findings": "count of clear CE violations",
      "medium_confidence_ce_disputes": "count of CE interpretation issues",
      "potential_ce_gaps": "count of CE catalog limitations",
      "total_ce_financial_impact": "dollar impact of CE findings",
      "ce_pricing_methodology_compliance": "carrier compliance assessment"
    },
    "overall_assessment": {
      "strongest_tier_1_arguments": ["top 3 highest confidence findings"],
      "key_tier_2_opportunities": ["top 3 medium confidence findings"],
      "tier_3_investigation_priorities": ["top 3 potential findings"],
      "total_financial_impact": "sum of all quantifiable findings",
      "recommended_immediate_actions": ["actions for next 30 days"],
      "recommended_strategic_approach": "overall case strategy",
      "success_probability_overall": "weighted average across all tiers"
    },
    "escalation_matrix": {
      "internal_expertise_required": ["categories requiring internal review"],
      "external_specialist_required": ["categories requiring outside experts"],
      "legal_counsel_required": ["categories requiring attorney involvement"],
      "immediate_action_items": ["findings requiring immediate response"],
      "strategic_planning_items": ["findings for long-term strategy"]
    }
  }
}
 
VALIDATION PROTOCOL:
□ All findings from previous stages analyzed and categorized
□ CE pricing integration properly assessed where applicable  
□ Success probabilities based on evidence strength and precedent
□ Action recommendations align with confidence levels
□ Timeline recommendations realistic and strategic
□ Financial impact calculations include CE pricing corrections
□ Escalation pathways clearly defined for each tier
□ Overall strategy coherent and actionable
 
This enhanced confidence framework provides tactical guidance while properly integrating CE pricing validation and ensuring consistent strategic recommendations across multiple estimate runs."""
    }

In [18]:
def load_llm_scopes_from_file(filepath):
    """Load LLM output from a text file containing JSON array."""
    try:
        with open(filepath, "r") as f:
            # Clean any stray markdown fencing or whitespace if needed
            raw = f.read().strip()
            # In case file is wrapped in ```json ... ```
            if raw.startswith("```json"):
                raw = raw.strip("```json").strip("```")
            data = json.loads(raw)
        print(f"✅ Loaded {len(data)} scope items from {filepath}")
        return data
    except Exception as e:
        print(f"❌ Failed to load file {filepath}: {e}")
        return []

def create_estimate_from_llm_output(llm_output, zipcode="78749", estimate_type=3):
    url = BASE_URL + "estimates/create"
    headers = {"x-api-key": API_KEY}

    estimate_array = [
        {"scope_id": item["scope_id"], "quantity": item["quantity"]}
        for item in llm_output
    ]

    payload = {
        "zipcode": zipcode,
        "estimatearray": estimate_array,
        "type": estimate_type
    }

    response = requests.post(url, headers=headers, json=payload)
    print("ESTIMATE REQUEST:", response.status_code)

    try:
        response_data = response.json()
        print(json.dumps(response_data, indent=2))
        return response_data
    except json.JSONDecodeError:
        print("Invalid JSON in response")
        return None


In [25]:
!pip3 install -U 'anthropic[vertex]'

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)

[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [38]:
import json
import requests
import os
# ---------- Async Gemini Runner ----------
from anthropic import AnthropicVertex

LOCATION = "us-east5"
client = AnthropicVertex(region=LOCATION, project_id="aistimate")

async def run_block(label, prompt, file_parts=None):
    contents = [{"role": "user", "content": prompt}]
    output = ""

    try:
        print(f"🔹 Running {label}...")
        resp = client.messages.create(
            model="claude-sonnet-4@20250514",
            max_tokens=4096,
            temperature=0,
            messages=contents
        )
        # resp.content is a list of TextBlock, ToolUseBlock, etc.
        output = "".join(block.text for block in resp.content if hasattr(block, "text"))
        print(f"✅ {label} complete ({len(output)} chars)")
    except Exception as e:
        output = f"[ERROR in {label}] {e}"
        print(output)

    return label, output







# ---------- Master Pipeline ----------

async def run_aistimate_pipeline(file_paths,pricing_path):
    prompts = build_prompts()

    # Assign files
    # 1) Carrier: only the first file
    carrier_parts = [make_part(file_paths[0])]

    # 2) Evidence: file_paths[0] plus file_paths[2:]
    evidence_paths = [file_paths[0]] + file_paths[2:]
    evidence_parts = [make_part(path) for path in evidence_paths]

    # 3) Policy: again, just the first file (if that’s what you meant)
    policy_parts = [make_part(path) for path in file_paths[1:2]]


    # Stage 1: Run code lookup & damage analysis in parallel
    stage1_tasks = [
        run_block("CODE_LOOKUP", prompts["CODE_LOOKUP"], carrier_parts),
        # run_block("REPORT_ANALYSIS", prompts["REPORT_ANALYSIS"], evidence_parts),
    ]
    stage1_results = await asyncio.gather(*stage1_tasks)
    context = {label: output for label, output in stage1_results}

    for label, content in stage1_results:
        save_output(label, content)

    # Stage 2: Scoping logic (needs prior outputs)
#     scoping_context = (
#         f"--- CODE LOOKUP ---n{context['CODE_LOOKUP']}nn"
#         f"--- DAMAGE OBSERVATIONS ---n{context['REPORT_ANALYSIS']}"
#     )
#     label, scoping_output = await run_block("SCOPING_LOGIC", prompts["SCOPING_LOGIC"] + "nn" + scoping_context)
#     save_output(label, scoping_output)
#     context["SCOPING_LOGIC"] = scoping_output

#     pricing_context = (
#         f"--- CODE LOOKUP ---n{context['CODE_LOOKUP']}nn"
#         f"--- DAMAGE OBSERVATIONS ---n{context['REPORT_ANALYSIS']}"
#         f"--- SCOPING LOGIC ---n{context['SCOPING_LOGIC']}"
#     )
    
#     label, pricing_output = await run_block("PRICING_LOGIC", prompts["PRICING_LOGIC"] + "nn" + pricing_context,policy_parts)
#     save_output(label, pricing_output)
#     filepath = os.path.join(pricing_path,"output_pricing_logic.txt")
#     print("Pricing output saved to:", filepath)
#     scope_data = load_llm_scopes_from_file(filepath)
#     if scope_data:
#     # Create estimate
#         valid_scopes = [
#         item for item in scope_data
#         if not item["scope_id"].startswith("MISSING_")
#         ]

# # Call your estimate creation function
#         estimate_response = create_estimate_from_llm_output(valid_scopes, zipcode="80210", estimate_type=2)
#         # print("Estimate response:", estimate_response)
#         # Save estimate_response to file in pricing_path directory
#         estimate_filepath = os.path.join(pricing_path, "estimate_response.json")
#         with open(estimate_filepath, "w") as f:
#             f.write(json.dumps(estimate_response, indent=2))
#         print("Estimate response saved to:", estimate_filepath)
        
    
    



# Use that structured version in the context
    # estimate_context = (
    # f"--- CODE MANDATES ---\n{context['CODE_LOOKUP']}\n\n"
    # f"--- DAMAGE FINDINGS ---\n{context['REPORT_ANALYSIS']}\n\n" 
    # f"--- SCOPING LOGIC ---\n{context['SCOPING_LOGIC']}\n\n"   
    # f"--- PRICING LOGIC ---\n{pricing_output}\n\n"
    # f"--- SELECTED SCOPES ---\n{estimate_response}\n"
    # )   

    #     # f"--- POLICY LOGIC ---n{context['POLICY_LOGIC_output']}nn"
    
    # label, estimate_output = await run_block("ESTIMATE", prompts["ESTIMATE"] + "nn" + estimate_context)
    # save_output(label, estimate_output)

    # # Stage 4: Rebuttal
    # # Stage 4: Rebuttal (pass carrier file for comparison)
    # label, rebuttal_output = await run_block(
    # "REBUTTAL",
    # prompts["REBUTTAL"] + "nn" + estimate_output,
    # file_parts=carrier_parts  # 🔹 passes carrier estimate as input context
    # )
    # save_output(label, rebuttal_output)


    return {
        "code_lookup": context["CODE_LOOKUP"],
        # "report_analysis": context["REPORT_ANALYSIS"],
        # "scoping_logic": context["SCOPING_LOGIC"],
        # "estimate_output": estimate_output,
        # "rebuttal_output": rebuttal_output,
    }


In [10]:
API_KEY = "emym6vnmxyo0d62vz3w1kaj"

# Step 2: Set the base URL (use sandbox for testing)
BASE_URL = "https://api.clearestimates.com/cepia/"

In [40]:
output_dir = f"outputs/Aug10/2500074/run2_sonnet_codeLookup"
def save_output(label: str, content: str):
   
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Carrier Estimate ($16,113.56) Insurance Carrier Estimate.pdf",
    # "scopes_for_llm.txt",
    # "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Eagleview Report - Grace Forensic Plaintiff Expert Estimate.PDF",
    # "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Forensic Damage Assessment - Grace Forensic Plaintiff Expert Estimate.pdf",
    # "Aiestimate/ 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Inspection Report For Primary Structure 04-16-2025 Plaintiff Expert Estimate_compressed.pdf",
],output_dir)

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (2135 chars)
📝 Saved: outputs/Aug10/2500074/run2_sonnet_codeLookup/output_code_lookup.txt
